In [4]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import pandas as pd
import numpy as np


In [5]:
# load data
data = pd.read_csv("dataset/HAM10000_metadata.csv")

In [6]:
import os
import glob

image_paths = {}
folders = ["dataset/HAM10000_images_part_1", "dataset/HAM10000_images_part_2"]
for folder in folders:
    for img_path in glob.glob(os.path.join(folder, "*.jpg")):
        image_id = os.path.splitext(os.path.basename(img_path))[0]
        image_paths[image_id] = img_path

print("Total images:", len(image_paths))
data["image_path"] = data["image_id"].map(image_paths)
print(data["image_path"].isnull().sum())

Total images: 10015
0


In [7]:
# tarin validation test-split
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    data,
    test_size=0.30,
    stratify=data["dx"],
    random_state=42
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["dx"],
    random_state=42
)

In [8]:
# handli class imbalance
from sklearn.utils.class_weight import compute_class_weight


classes = np.unique(train_df["dx"])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["dx"]
)

class_weights = dict(enumerate(weights))
print(class_weights)

{0: np.float64(4.37305053025577), 1: np.float64(2.7817460317460316), 2: np.float64(1.3022478172023035), 3: np.float64(12.36331569664903), 4: np.float64(1.285530900421786), 5: np.float64(0.21338772031292808), 6: np.float64(10.115440115440116)}


In [10]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,

    rotation_range=20,

    width_shift_range=0.1,
    height_shift_range=0.1,

    horizontal_flip=True,
    vertical_flip=True,

    zoom_range=0.2,

    brightness_range=[0.8,1.2]
)

# Validation Generator
val_datagen = ImageDataGenerator(
    rescale=1./255
)
# Test Generator
test_datagen = ImageDataGenerator(
    rescale=1./255
)

In [11]:
# create train  pipeline
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,

    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    shuffle=True
)
# create validatation pipeline
val_generator = val_datagen.flow_from_dataframe(
    dataframe=val_df,

    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    shuffle=False
)

# create test pipeline
test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,

    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    shuffle=False
)

Found 7010 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.


## DROPOUT

In [1]:
Dropout_results = []
Dropout_results.append({
    "Model": "Dropout CNN",
    "Train Accuracy": 47.97,
    "Validation Accuracy": 48.74,
    "Test Accuracy": 46.64,
    "Train Loss": 1.3525,
    "Validation Loss": 1.2767,
    "Test Loss": 1.3052
})

In [ ]:
##LEARNINGRATE, EARLYSTOPPING

In [18]:
# CNN DROOUT
from tensorflow.keras.layers import Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Dropout,
    Flatten,
    Dense
)

from tensorflow.keras.optimizers import (
    Adam,
    SGD,
    RMSprop
)

import keras_tuner as kt

In [19]:
cnn_dropout = Sequential(name="CNN_Dropout")

# Block 1
cnn_dropout.add(Conv2D(32, (3,3), activation="relu", input_shape=(224,224,3)))
cnn_dropout.add(MaxPooling2D((2,2)))
cnn_dropout.add(Dropout(0.25))

# Block 2
cnn_dropout.add(Conv2D(64, (3,3), activation="relu"))
cnn_dropout.add(MaxPooling2D((2,2)))
cnn_dropout.add(Dropout(0.25))

# Block 3
cnn_dropout.add(Conv2D(128, (3,3), activation="relu"))
cnn_dropout.add(MaxPooling2D((2,2)))
cnn_dropout.add(Dropout(0.25))

# Block 4
cnn_dropout.add(Conv2D(256, (3,3), activation="relu"))
cnn_dropout.add(MaxPooling2D((2,2)))
cnn_dropout.add(Dropout(0.25))

# Fully Connected Layer
cnn_dropout.add(Flatten())
cnn_dropout.add(Dense(256, activation="relu"))
cnn_dropout.add(Dropout(0.5))

# Output Layer
cnn_dropout.add(Dense(7, activation="softmax"))

cnn_dropout.summary()

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "CNN_Dropout"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 36864)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     9,437,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,827,655 (37.49 MB)

 Trainable params: 9,827,655 (37.49 MB)

 Non-trainable params: 0 (0.00 B)

In [20]:
from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

In [21]:
cnn_dropout.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [22]:
history_dropout = cnn_dropout.fit(
    train_generator,
    validation_data=val_generator,
    epochs=8,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 200s 906ms/step - accuracy: 0.2312 - loss: 1.9666 - val_accuracy: 0.5020 - val_loss: 1.8305
Epoch 2/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 155s 704ms/step - accuracy: 0.3994 - loss: 1.8793 - val_accuracy: 0.4667 - val_loss: 1.6664
Epoch 3/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 144s 654ms/step - accuracy: 0.3961 - loss: 1.8210 - val_accuracy: 0.4720 - val_loss: 1.4929
Epoch 4/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 144s 655ms/step - accuracy: 0.4118 - loss: 1.7748 - val_accuracy: 0.4747 - val_loss: 1.4740
Epoch 5/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 143s 647ms/step - accuracy: 0.4321 - loss: 1.6836 - val_accuracy: 0.5559 - val_loss: 1.3201
Epoch 6/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 145s 657ms/step - accuracy: 0.4514 - loss: 1.5147 - val_accuracy: 0.4787 - val_loss: 1.2716
Epoch 7/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 140s 638ms/step - accuracy: 0.4553 - loss: 1.4903 - val_accuracy: 0.5613 - val_loss: 1.1799
Epoch 8/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 137s 623ms/step - accuracy: 0.4685 - loss: 1

In [23]:
train_loss, train_acc = cnn_dropout.evaluate(train_generator)
val_loss, val_acc = cnn_dropout.evaluate(val_generator)
test_loss, test_acc = cnn_dropout.evaluate(test_generator)

220/220 ━━━━━━━━━━━━━━━━━━━━ 54s 247ms/step - accuracy: 0.5496 - loss: 1.1839
47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 159ms/step - accuracy: 0.5453 - loss: 1.1545
47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 159ms/step - accuracy: 0.5422 - loss: 1.1801


In [24]:
cnn_dropout.save("models2/cnn_dropout_earlystopping.keras")
print("saved cnn_dropout")

saved cnn_dropout


In [2]:

Dropout_results.append({
    "Model": "Dropout CNN LR_ES",
    "Train Accuracy": 54.96,
    "Validation Accuracy": 54.53,
    "Test Accuracy": 54.22,
    "Train Loss": 1.1839,
    "Validation Loss": 1.1545,
    "Test Loss": 1.1801
})

## rmsprop

In [25]:
cnn_dropout_RMS = Sequential(name="CNN_Dropout")

# Block 1
cnn_dropout_RMS.add(Conv2D(32, (3,3), activation="relu", input_shape=(224,224,3)))
cnn_dropout_RMS.add(MaxPooling2D((2,2)))
cnn_dropout_RMS.add(Dropout(0.25))

# Block 2
cnn_dropout_RMS.add(Conv2D(64, (3,3), activation="relu"))
cnn_dropout_RMS.add(MaxPooling2D((2,2)))
cnn_dropout_RMS.add(Dropout(0.25))

# Block 3
cnn_dropout_RMS.add(Conv2D(128, (3,3), activation="relu"))
cnn_dropout_RMS.add(MaxPooling2D((2,2)))
cnn_dropout_RMS.add(Dropout(0.25))

# Block 4
cnn_dropout_RMS.add(Conv2D(256, (3,3), activation="relu"))
cnn_dropout_RMS.add(MaxPooling2D((2,2)))
cnn_dropout_RMS.add(Dropout(0.25))

# Fully Connected Layer
cnn_dropout_RMS.add(Flatten())
cnn_dropout_RMS.add(Dense(256, activation="relu"))
cnn_dropout_RMS.add(Dropout(0.5))

# Output Layer
cnn_dropout_RMS.add(Dense(7, activation="softmax"))

cnn_dropout_RMS.summary()

Model: "CNN_Dropout"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 36864)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │     9,437,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,827,655 (37.49 MB)

 Trainable params: 9,827,655 (37.49 MB)

 Non-trainable params: 0 (0.00 B)

In [26]:
from tensorflow.keras.optimizers import RMSprop
cnn_dropout_RMS.compile(
    optimizer=RMSprop(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [27]:
history_cnn_dropout_RMS = cnn_dropout_RMS.fit(
    train_generator,
    validation_data=val_generator,
    epochs=8,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 139s 627ms/step - accuracy: 0.3927 - loss: 1.9883 - val_accuracy: 0.5866 - val_loss: 1.8756
Epoch 2/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 139s 631ms/step - accuracy: 0.5044 - loss: 1.8774 - val_accuracy: 0.6218 - val_loss: 1.4574
Epoch 3/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 135s 614ms/step - accuracy: 0.4255 - loss: 1.8241 - val_accuracy: 0.6185 - val_loss: 1.2911
Epoch 4/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 135s 612ms/step - accuracy: 0.4013 - loss: 1.8046 - val_accuracy: 0.6491 - val_loss: 1.2223
Epoch 5/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 135s 613ms/step - accuracy: 0.4220 - loss: 1.7852 - val_accuracy: 0.6385 - val_loss: 1.2002
Epoch 6/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 137s 624ms/step - accuracy: 0.4121 - loss: 1.7814 - val_accuracy: 0.6225 - val_loss: 1.3407
Epoch 7/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 136s 619ms/step - accuracy: 0.4347 - loss: 1.7551 - val_accuracy: 0.6365 - val_loss: 1.3076
Epoch 8/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 135s 613ms/step - accuracy: 0.4508 - loss: 1

In [28]:
train_loss, train_acc = cnn_dropout_RMS.evaluate(train_generator)
val_loss, val_acc = cnn_dropout_RMS.evaluate(val_generator)
test_loss, test_acc = cnn_dropout_RMS.evaluate(test_generator)

220/220 ━━━━━━━━━━━━━━━━━━━━ 55s 249ms/step - accuracy: 0.5397 - loss: 1.2339
47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 159ms/step - accuracy: 0.5859 - loss: 1.1845
47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 159ms/step - accuracy: 0.6088 - loss: 1.1949


In [29]:
cnn_dropout_RMS.save("models2/cnn_dropout_RMS.keras")
print("saved cnn_dropout")

saved cnn_dropout


In [3]:

Dropout_results.append({
    "Model": "Dropout CNN(RMSprop)",
    "Train Accuracy": 53.97,
    "Validation Accuracy": 58.59,
    "Test Accuracy": 60.88,
    "Train Loss": 1.2339,
    "Validation Loss": 1.1845,
    "Test Loss": 1.1949
})

### SGD

In [30]:
cnn_dropout_sgd = Sequential(name="CNN_Dropout")

# Block 1
cnn_dropout_sgd.add(Conv2D(32, (3,3), activation="relu", input_shape=(224,224,3)))
cnn_dropout_sgd.add(MaxPooling2D((2,2)))
cnn_dropout_sgd.add(Dropout(0.25))

# Block 2
cnn_dropout_sgd.add(Conv2D(64, (3,3), activation="relu"))
cnn_dropout_sgd.add(MaxPooling2D((2,2)))
cnn_dropout_sgd.add(Dropout(0.25))

# Block 3
cnn_dropout_sgd.add(Conv2D(128, (3,3), activation="relu"))
cnn_dropout_sgd.add(MaxPooling2D((2,2)))
cnn_dropout_sgd.add(Dropout(0.25))

# Block 4
cnn_dropout_sgd.add(Conv2D(256, (3,3), activation="relu"))
cnn_dropout_sgd.add(MaxPooling2D((2,2)))
cnn_dropout_sgd.add(Dropout(0.25))

# Fully Connected Layer
cnn_dropout_sgd.add(Flatten())
cnn_dropout_sgd.add(Dense(256, activation="relu"))
cnn_dropout_sgd.add(Dropout(0.5))

# Output Layer
cnn_dropout_sgd.add(Dense(7, activation="softmax"))

cnn_dropout_sgd.summary()

Model: "CNN_Dropout"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_8 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 36864)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │     9,437,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,827,655 (37.49 MB)

 Trainable params: 9,827,655 (37.49 MB)

 Non-trainable params: 0 (0.00 B)

In [31]:
from tensorflow.keras.optimizers import SGD
 
cnn_dropout_sgd.compile(
    optimizer=SGD(learning_rate=0.001, momentum=0.9),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [32]:
history_cnn_dropout_sgd = cnn_dropout_sgd.fit(
    train_generator,
    validation_data=val_generator,
    epochs=8,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 136s 616ms/step - accuracy: 0.1792 - loss: 1.9704 - val_accuracy: 0.1145 - val_loss: 1.9382
Epoch 2/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 136s 616ms/step - accuracy: 0.2807 - loss: 1.9343 - val_accuracy: 0.2590 - val_loss: 1.9458
Epoch 3/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 134s 607ms/step - accuracy: 0.3974 - loss: 1.9189 - val_accuracy: 0.5646 - val_loss: 1.7819
Epoch 4/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 134s 607ms/step - accuracy: 0.3999 - loss: 1.8624 - val_accuracy: 0.0186 - val_loss: 1.9790
Epoch 5/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 133s 606ms/step - accuracy: 0.3230 - loss: 1.8784 - val_accuracy: 0.5193 - val_loss: 1.6972
Epoch 6/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 164s 743ms/step - accuracy: 0.3930 - loss: 1.8317 - val_accuracy: 0.5519 - val_loss: 1.5337
Epoch 7/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 1066s 5s/step - accuracy: 0.3876 - loss: 1.7709 - val_accuracy: 0.5573 - val_loss: 1.5186
Epoch 8/8
220/220 ━━━━━━━━━━━━━━━━━━━━ 133s 603ms/step - accuracy: 0.3700 - loss: 1.7

In [35]:
train_loss, train_acc = cnn_dropout_sgd.evaluate(train_generator)
val_loss, val_acc = cnn_dropout_sgd.evaluate(val_generator)
test_loss, test_acc = cnn_dropout_sgd.evaluate(test_generator)

220/220 ━━━━━━━━━━━━━━━━━━━━ 54s 247ms/step - accuracy: 0.5837 - loss: 1.4187
47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 159ms/step - accuracy: 0.6025 - loss: 1.3900
47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 160ms/step - accuracy: 0.6035 - loss: 1.4062


In [36]:
cnn_dropout_sgd.save("models2/cnn_dropout_sgd.keras")
print("saved cnn_dropout_sgd")

saved cnn_dropout_sgd


In [4]:

Dropout_results.append({
    "Model": "Dropout CNN(SGD)",
    "Train Accuracy": 58.37,
    "Validation Accuracy": 60.25,
    "Test Accuracy": 60.35,
    "Train Loss": 1.4187,
    "Validation Loss": 1.3900,
    "Test Loss": 1.4062
})

## Batchsize(64)

In [37]:
# create train  pipeline
train_generator_64 = train_datagen.flow_from_dataframe(
    dataframe=train_df,

    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=64,
    class_mode="categorical",
    shuffle=True
)
# create validatation pipeline
val_generator_64 = val_datagen.flow_from_dataframe(
    dataframe=val_df,

    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=64,
    class_mode="categorical",
    shuffle=False
)

# create test pipeline
test_generator_64 = test_datagen.flow_from_dataframe(
    dataframe=test_df,

    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=64,
    class_mode="categorical",
    shuffle=False
)

Found 7010 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.


In [38]:
cnn_dropout_16 = Sequential(name="CNN_Dropout")

# Block 1
cnn_dropout_16.add(Conv2D(32, (3,3), activation="relu", input_shape=(224,224,3)))
cnn_dropout_16.add(MaxPooling2D((2,2)))
cnn_dropout_16.add(Dropout(0.25))

# Block 2
cnn_dropout_16.add(Conv2D(64, (3,3), activation="relu"))
cnn_dropout_16.add(MaxPooling2D((2,2)))
cnn_dropout_16.add(Dropout(0.25))

# Block 3
cnn_dropout_16.add(Conv2D(128, (3,3), activation="relu"))
cnn_dropout_16.add(MaxPooling2D((2,2)))
cnn_dropout_16.add(Dropout(0.25))

# Block 4
cnn_dropout_16.add(Conv2D(256, (3,3), activation="relu"))
cnn_dropout_16.add(MaxPooling2D((2,2)))
cnn_dropout_16.add(Dropout(0.25))

# Fully Connected Layer
cnn_dropout_16.add(Flatten())
cnn_dropout_16.add(Dense(256, activation="relu"))
cnn_dropout_16.add(Dropout(0.5))

# Output Layer
cnn_dropout_16.add(Dense(7, activation="softmax"))

cnn_dropout_16.summary()

/Users/aximsoft/tfenv/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "CNN_Dropout"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_12 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_15 (Conv2D)              │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_15 (MaxPooling2D) │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_18 (Dropout)            │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 36864)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │     9,437,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_19 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,827,655 (37.49 MB)

 Trainable params: 9,827,655 (37.49 MB)

 Non-trainable params: 0 (0.00 B)

In [39]:
cnn_dropout_16.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [40]:
history_cnn_dropout_16= cnn_dropout_16.fit(
    train_generator_64,
    validation_data=val_generator_64,
    epochs=8,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 139s 1s/step - accuracy: 0.3350 - loss: 1.9389 - val_accuracy: 0.5333 - val_loss: 1.6927
Epoch 2/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 139s 1s/step - accuracy: 0.4080 - loss: 1.8552 - val_accuracy: 0.5113 - val_loss: 1.6324
Epoch 3/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 137s 1s/step - accuracy: 0.4284 - loss: 1.8076 - val_accuracy: 0.5939 - val_loss: 1.3507
Epoch 4/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 136s 1s/step - accuracy: 0.4023 - loss: 1.7840 - val_accuracy: 0.5493 - val_loss: 1.4078
Epoch 5/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 143s 1s/step - accuracy: 0.4228 - loss: 1.7434 - val_accuracy: 0.4947 - val_loss: 1.4983
Epoch 6/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 149s 1s/step - accuracy: 0.4103 - loss: 1.7238 - val_accuracy: 0.5226 - val_loss: 1.5180


In [41]:
train_loss, train_acc = cnn_dropout_16.evaluate(train_generator)
val_loss, val_acc = cnn_dropout_16.evaluate(val_generator)
test_loss, test_acc = cnn_dropout_16.evaluate(test_generator)

220/220 ━━━━━━━━━━━━━━━━━━━━ 60s 270ms/step - accuracy: 0.5893 - loss: 1.3837
47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 162ms/step - accuracy: 0.5939 - loss: 1.3507
47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 170ms/step - accuracy: 0.6028 - loss: 1.3625


In [42]:
cnn_dropout_16.save("models2/cnn_dropout_16.keras")
print("saved cnn_dropout_16")

saved cnn_dropout_16


In [10]:

Dropout_results.append({
    "Model": "Dropout CNN Batchsize(64)",
    "Train Accuracy": 58.93,
    "Validation Accuracy": 59.39,
    "Test Accuracy": 60.28,
    "Train Loss": 1.3837,
    "Validation Loss": 1.3507,
    "Test Loss": 1.3625
})

In [ ]:
# HyperParameter

In [43]:
import keras_tuner as kt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam, SGD, RMSprop

IMG_SIZE = 224
NUM_CLASSES = 7


def build_cnn_dropout(hp):

    model = Sequential(name="CNN_Dropout_Hyperparameter")
    model.add(
        Conv2D(
            filters=hp.Choice("filters1", [32, 64]),
            kernel_size=(3,3),
            activation="relu",
            input_shape=(IMG_SIZE, IMG_SIZE, 3)
        )
    )

    model.add(MaxPooling2D((2,2)))

    model.add(
        Dropout(
            hp.Choice("dropout1",[0.2,0.25,0.3])
        )
    )
    model.add(
        Conv2D(
            filters=hp.Choice("filters2",[64,128]),
            kernel_size=(3,3),
            activation="relu"
        )
    )

    model.add(MaxPooling2D((2,2)))

    model.add(
        Dropout(
            hp.Choice("dropout2",[0.2,0.25,0.3])
        )
    )

    model.add(
        Conv2D(
            filters=hp.Choice("filters3",[128,256]),
            kernel_size=(3,3),
            activation="relu"
        )
    )

    model.add(MaxPooling2D((2,2)))

    model.add(
        Dropout(
            hp.Choice("dropout3",[0.2,0.25,0.3])
        )
    )
    model.add(
        Conv2D(
            filters=hp.Choice("filters4",[256,512]),
            kernel_size=(3,3),
            activation="relu"
        )
    )

    model.add(MaxPooling2D((2,2)))
    model.add(
        Dropout(
            hp.Choice("dropout4",[0.2,0.25,0.3])
        )
    )
    model.add(Flatten())

    model.add(
        Dense(
            units=hp.Choice("dense_units",[128,256,512]),
            activation="relu"
        )
    )

    model.add(
        Dropout(
            hp.Choice("dense_dropout",[0.4,0.5,0.6])
        )
    )
    model.add(Dense(NUM_CLASSES, activation="softmax"))

    optimizer = hp.Choice(
        "optimizer",
        ["adam","sgd","rmsprop"]
    )

    learning_rate = hp.Choice(
        "learning_rate",
        [1e-2,1e-3,1e-4]
    )

    if optimizer=="adam":
        opt=Adam(learning_rate=learning_rate)

    elif optimizer=="sgd":
        opt=SGD(
            learning_rate=learning_rate,
            momentum=0.9
        )

    else:
        opt=RMSprop(learning_rate=learning_rate)

    model.compile(
        optimizer=opt,
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [45]:
tuner = kt.RandomSearch(
    build_cnn_dropout,
    objective="val_accuracy",
    max_trials=3,
    directory="tuner_results",
    project_name="cnn_dropout"
)

tuner.search(
    train_generator,
    validation_data=val_generator,
    epochs=7
)

Trial 3 Complete [00h 42m 10s]
val_accuracy: 0.6697736382484436

Best val_accuracy So Far: 0.6704394221305847
Total elapsed time: 01h 38m 52s


In [47]:
best_hp = tuner.get_best_hyperparameters(1)[0]

print(best_hp.values)

{'filters1': 32, 'dropout1': 0.3, 'filters2': 64, 'dropout2': 0.3, 'filters3': 128, 'dropout3': 0.3, 'filters4': 256, 'dropout4': 0.25, 'dense_units': 128, 'dense_dropout': 0.6, 'optimizer': 'adam', 'learning_rate': 0.001}


In [48]:
best_model = tuner.hypermodel.build(best_hp)

history = best_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=6
)

Epoch 1/6
220/220 ━━━━━━━━━━━━━━━━━━━━ 136s 612ms/step - accuracy: 0.6648 - loss: 1.0973 - val_accuracy: 0.6698 - val_loss: 1.0234
Epoch 2/6
220/220 ━━━━━━━━━━━━━━━━━━━━ 134s 606ms/step - accuracy: 0.6692 - loss: 1.0140 - val_accuracy: 0.6698 - val_loss: 0.9528
Epoch 3/6
220/220 ━━━━━━━━━━━━━━━━━━━━ 137s 620ms/step - accuracy: 0.6698 - loss: 0.9839 - val_accuracy: 0.6698 - val_loss: 1.0524
Epoch 4/6
220/220 ━━━━━━━━━━━━━━━━━━━━ 133s 605ms/step - accuracy: 0.6705 - loss: 0.9662 - val_accuracy: 0.6704 - val_loss: 0.9021
Epoch 5/6
220/220 ━━━━━━━━━━━━━━━━━━━━ 134s 609ms/step - accuracy: 0.6718 - loss: 0.9491 - val_accuracy: 0.6724 - val_loss: 0.8992
Epoch 6/6
220/220 ━━━━━━━━━━━━━━━━━━━━ 133s 604ms/step - accuracy: 0.6739 - loss: 0.9151 - val_accuracy: 0.6724 - val_loss: 0.9279


In [49]:
best_model.save("models/cnn_dropout_hyperparameter.keras")

In [50]:
train_loss, train_acc = best_model.evaluate(train_generator)
val_loss, val_acc = best_model.evaluate(val_generator)
test_loss, test_acc = best_model.evaluate(test_generator)

220/220 ━━━━━━━━━━━━━━━━━━━━ 57s 258ms/step - accuracy: 0.6726 - loss: 0.9371
47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 158ms/step - accuracy: 0.6724 - loss: 0.9279
47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 158ms/step - accuracy: 0.6713 - loss: 0.9323


In [5]:

Dropout_results.append({
    "Model": "Dropout HyperParameter",
    "Train Accuracy": 67.26,
    "Validation Accuracy": 67.64,
    "Test Accuracy": 67.13,
    "Train Loss": 0.9371,
    "Validation Loss": 0.9371,
    "Test Loss": 0.9323
})

In [8]:
import pandas as pd


In [11]:
comparison_dropout_cnn = pd.DataFrame(Dropout_results)
comparison_dropout_cnn

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,Train Loss,Validation Loss,Test Loss
0,Dropout CNN,47.97,48.74,46.64,1.3525,1.2767,1.3052
1,Dropout CNN LR_ES,54.96,54.53,54.22,1.1839,1.1545,1.1801
2,Dropout CNN(RMSprop),53.97,58.59,60.88,1.2339,1.1845,1.1949
3,Dropout CNN(SGD),58.37,60.25,60.35,1.4187,1.3900,1.4062
4,Dropout HyperParameter,67.26,67.64,67.13,0.9371,0.9371,0.9323
5,Dropout CNN Batchsize(64),58.93,59.39,60.28,1.3837,1.3507,1.3625


In [12]:
comparison_dropout_cnn.to_csv("dataset/dropout_comparison.csv", index=False)
print("saved")

saved
